# 03 - Cleaning (DS, with DA)

Reads `data/raw/` (never modified) and writes cleaned tables to `data/clean/`. Run it top to bottom; the same input always gives the same output.

**Steps**
1. Lead source: 28 spellings -> 14 channels
2. Country: labels -> ISO codes
3. Budgets: flag and fix obvious typos
4. Duplicates: drop duplicate customers and re-point their leads, sessions and responses to the surviving `customer_id`
5. Validate and write

**Before you share any output:** agree `cleaning_maps.py` and `outputs/duplicate_customer_map.csv` with the DA team. They are building `clean_*` views in PostgreSQL (Day 4) and the two versions must give the same counts.

Needs `outputs/duplicate_customer_map.csv` from notebook 02 (section B6). Without it, step 4 is skipped.

## 0. Setup

In [1]:
from pathlib import Path
import re, sys
import numpy as np
import pandas as pd
import duckdb

sys.path.insert(0, str(Path.cwd()))
from cleaning_maps import LEAD_SOURCE_MAP, COUNTRY_MAP

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

ROOT = Path.cwd().parent
base = ROOT / "data" / "raw" / "veltrix_medium"
CLEAN = ROOT / "data" / "clean"
OUT = ROOT / "notebooks" / "outputs"
CLEAN.mkdir(parents=True, exist_ok=True)
assert base.exists(), f"Data folder not found: {base}"

SMALL = ["agents", "campaigns", "campaign_responses", "customers", "interactions", "leads",
         "locations", "payments", "properties", "site_visits", "transactions"]
BIG = ["web_sessions", "property_views"]

raw = {t: pd.read_parquet(base / f"{t}.parquet") for t in SMALL}
clean = {t: df.copy() for t, df in raw.items()}      # work on copies; raw stays untouched
log = []                                              # one row per change, for the DA data quality log

def note(step, table, column, rows_changed, detail=""):
    log.append({"step": step, "table": table, "column": column, "rows_changed": int(rows_changed), "detail": detail})
    print(f"[{step}] {table}.{column}: {int(rows_changed)} rows {detail}")

{t: df.shape for t, df in raw.items()}

{'agents': (300, 6),
 'campaigns': (270, 10),
 'campaign_responses': (808953, 5),
 'customers': (102000, 19),
 'interactions': (532789, 8),
 'leads': (95804, 11),
 'locations': (36, 9),
 'payments': (106199, 10),
 'properties': (16667, 19),
 'site_visits': (233593, 6),
 'transactions': (16665, 14)}

## 1. Lead source

In [2]:
leads = clean["leads"]
unmapped = sorted(set(leads["lead_source"].dropna().unique()) - set(LEAD_SOURCE_MAP))
assert not unmapped, f"Add these lead_source values to LEAD_SOURCE_MAP in cleaning_maps.py: {unmapped}"

before = leads["lead_source"].copy()
leads["lead_source"] = before.map(LEAD_SOURCE_MAP)
note("lead_source", "leads", "lead_source", (before != leads["lead_source"]).sum(),
     f"re-labelled; {before.nunique()} raw values -> {leads['lead_source'].nunique()} channels")
leads["lead_source"].value_counts()

[lead_source] leads.lead_source: 8789 rows re-labelled; 28 raw values -> 14 channels


lead_source
Rightmove                 25243
Zoopla                    12841
Veltrix website           12275
Google Ads                 6425
Referral                   5972
Email newsletter           5622
Facebook                   5413
OnTheMarket                4532
TikTok                     4481
Instagram                  4351
Branch walk-in             3008
Overseas agent partner     2575
Relocation partner         1943
WeChat                     1123
Name: count, dtype: int64

## 2. Country
`country_of_residence` -> ISO code in a new column `country_iso`. The original label is kept.
If a label is missing from `COUNTRY_MAP`, the cell stops and lists it. Add it in `cleaning_maps.py`, then re-run.

In [3]:
cust = clean["customers"]
labels = cust["country_of_residence"].dropna().unique()
unmapped = sorted(set(labels) - set(COUNTRY_MAP))
if unmapped:
    print("Labels NOT in COUNTRY_MAP (add them to cleaning_maps.py):")
    print(cust["country_of_residence"].value_counts().loc[unmapped].to_string())
assert not unmapped, "Fix COUNTRY_MAP first"

cust["country_iso"] = cust["country_of_residence"].map(COUNTRY_MAP)
note("country", "customers", "country_iso", cust["country_iso"].notna().sum(), f"-> {cust['country_iso'].nunique()} ISO codes")
display(cust["country_iso"].value_counts().to_frame("customers"))

# Consistency check: is_overseas should mean "not GB"
display(pd.crosstab(cust["is_overseas"], cust["country_iso"].eq("GB"), rownames=["is_overseas"], colnames=["country_iso == GB"]))

[country] customers.country_iso: 102000 rows -> 12 ISO codes


,customers
country_iso,
GB,92404
HK,1862
AE,1638
US,1230
CN,903
IN,863
SG,859
NG,787
CA,545


country_iso == GB,False,True
is_overseas,,
False,0,92404
True,9596,0


## 3. Budgets
`stated_budget_gbp` (purchase) and `stated_rent_budget_pcm_gbp` (rent per month) are separate columns, each with its own scale. Notebook 02 showed two kinds of slip:
- **Entered in thousands** (purchase only): values like 240, 360, 470, 880 where a purchase budget should be in the hundreds of thousands. Multiply by 1,000.
- **Extra zero**: purchase budgets of 2.85M to 9.2M, rent budgets of 12,000 to 27,000 per month. Divide by 10.

Each rule is set per column in `RULES`. The original column is kept, and a `_typo_fix` column records which rule fired (blank = untouched).
**Review the flagged rows before you accept this.** A genuinely wealthy buyer above the purchase threshold would be wrongly divided by 10.

In [4]:
RULES = {   # column: (multiply by 1000 if below, divide by 10 if at least this many times the median)
    "stated_budget_gbp":          {"thousands_below": 1_000, "extra_zero_ratio": 8},
    "stated_rent_budget_pcm_gbp": {"thousands_below": None,  "extra_zero_ratio": 5},
}
BUDGET_COLS = [c for c in cust.columns if re.search("budget", c, re.I) and pd.api.types.is_numeric_dtype(cust[c])]
print("Budget columns found:", BUDGET_COLS)
unruled = [c for c in BUDGET_COLS if c not in RULES]
if unruled:
    print("No rule set for:", unruled, "(left untouched; add them to RULES)")

for c, rule in RULES.items():
    if c not in cust.columns:
        continue
    s = cust[c].astype("float64")
    med = s.median()
    ratio = s / med
    in_thousands = (s < rule["thousands_below"]) if rule["thousands_below"] else pd.Series(False, index=s.index)
    extra_zero = ratio >= rule["extra_zero_ratio"]
    fixed = np.where(in_thousands, s * 1000, np.where(extra_zero, s / 10, s))
    cust[f"{c}_clean"] = pd.array(np.where(np.isnan(fixed), np.nan, np.round(fixed)), dtype="Float64").astype("Int64")
    cust[f"{c}_typo_fix"] = np.where(in_thousands, "x1000", np.where(extra_zero, "div10", ""))
    n_fix = int((in_thousands | extra_zero).sum())
    note("budget", "customers", f"{c}_clean", n_fix,
         f"fixed ({int(in_thousands.sum())} in thousands, {int(extra_zero.sum())} extra zero) = {n_fix / s.notna().sum() * 100:.2f}% of non-missing (README: about 0.4% are typos)")
    display(cust.loc[cust[f"{c}_typo_fix"] != "", [c, f"{c}_clean", f"{c}_typo_fix"]].sort_values(c).head(12))

Budget columns found: ['stated_budget_gbp', 'stated_rent_budget_pcm_gbp']
[budget] customers.stated_budget_gbp_clean: 119 rows fixed (64 in thousands, 55 extra zero) = 0.24% of non-missing (README: about 0.4% are typos)


,stated_budget_gbp,stated_budget_gbp_clean,stated_budget_gbp_typo_fix
21571,110,110000,x1000
41010,115,115000,x1000
99043,120,120000,x1000
36550,155,155000,x1000
68711,160,160000,x1000
95528,160,160000,x1000
9483,165,165000,x1000
56940,180,180000,x1000
73328,180,180000,x1000
14362,190,190000,x1000


[budget] customers.stated_rent_budget_pcm_gbp_clean: 54 rows fixed (0 in thousands, 54 extra zero) = 0.41% of non-missing (README: about 0.4% are typos)


,stated_rent_budget_pcm_gbp,stated_rent_budget_pcm_gbp_clean,stated_rent_budget_pcm_gbp_typo_fix
54589,7550,755,div10
43073,8150,815,div10
48797,8300,830,div10
27351,9000,900,div10
39529,9000,900,div10
76473,9000,900,div10
51271,9600,960,div10
91931,10000,1000,div10
97964,10200,1020,div10
707,10500,1050,div10


## 4. Duplicate customers
Drops the duplicate rows from `customers`, and re-points `customer_id` in every table that has it, so the duplicate's leads, sessions and responses belong to the surviving customer.
Leads are **not** deleted: a survivor now has the duplicate's extra lead as well, which is expected.

In [5]:
map_path = OUT / "duplicate_customer_map.csv"
if map_path.exists():
    dup = pd.read_csv(map_path)
else:
    dup = pd.DataFrame(columns=["duplicate_id", "survivor_id"])
    print("No duplicate map found. Run section B of notebook 02 first. Skipping de-duplication.")

# a survivor must never itself be listed as a duplicate
assert not set(dup["survivor_id"]) & set(dup["duplicate_id"]), "Map has chains (a survivor that is also a duplicate)"
remap = dict(zip(dup["duplicate_id"], dup["survivor_id"]))
print(f"{len(remap)} duplicate ids -> {dup['survivor_id'].nunique()} survivors")

for t, df in clean.items():
    if "customer_id" in df.columns and t != "customers":
        before = df["customer_id"].copy()
        df["customer_id"] = before.map(remap).fillna(before)
        note("dedupe", t, "customer_id", (before != df["customer_id"]).sum(), "re-pointed to survivor")

n0 = len(cust)
clean["customers"] = cust = cust[~cust["customer_id"].isin(remap)].reset_index(drop=True)
note("dedupe", "customers", "(rows)", n0 - len(cust), "duplicate rows dropped")

1427 duplicate ids -> 1427 survivors
[dedupe] campaign_responses.customer_id: 2260 rows re-pointed to survivor
[dedupe] leads.customer_id: 1407 rows re-pointed to survivor
[dedupe] transactions.customer_id: 230 rows re-pointed to survivor
[dedupe] customers.(rows): 1427 rows duplicate rows dropped


## 5. Write the clean tables
Small tables are written with pandas. `web_sessions` and `property_views` are 2M and 15M rows, so DuckDB does the re-pointing and writes them straight to parquet.

In [6]:
for t, df in clean.items():
    df.to_parquet(CLEAN / f"{t}.parquet", index=False)

con = duckdb.connect()
D = base.as_posix()
dmap = (OUT / "duplicate_customer_map.csv").as_posix() if map_path.exists() else None
if dmap:
    con.execute(f"CREATE OR REPLACE TABLE dupmap AS SELECT duplicate_id, survivor_id FROM read_csv_auto('{dmap}')")
else:
    con.execute("CREATE OR REPLACE TABLE dupmap (duplicate_id VARCHAR, survivor_id VARCHAR)")

for t in BIG:
    con.execute(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM read_parquet('{D}/{t}/*.parquet')")
    cols = con.sql(f"DESCRIBE {t}").df()["column_name"].tolist()
    target = (CLEAN / f"{t}.parquet").as_posix()
    if "customer_id" in cols:
        con.execute(f'''COPY (SELECT x.* REPLACE (COALESCE(m.survivor_id, x.customer_id) AS customer_id)
                          FROM {t} x LEFT JOIN dupmap m ON x.customer_id = m.duplicate_id)
                        TO '{target}' (FORMAT PARQUET)''')
        n = con.sql(f"SELECT COUNT(*) AS n FROM {t} x JOIN dupmap m ON x.customer_id = m.duplicate_id").df().iloc[0, 0]
        note("dedupe", t, "customer_id", n, "re-pointed to survivor")
    else:
        con.execute(f"COPY (SELECT * FROM {t}) TO '{target}' (FORMAT PARQUET)")
    print(f"wrote {target}")

pd.DataFrame(log).to_csv(OUT / "cleaning_log.csv", index=False)
print("\nWrote", len(clean), "small tables + 2 large tables to", CLEAN)

[dedupe] web_sessions.customer_id: 2237 rows re-pointed to survivor
wrote c:/Users/pc/Documents/AMDARI INTERNSHIP/Veltrix Segmentation/veltrix-segmentation_A/data/clean/web_sessions.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

wrote c:/Users/pc/Documents/AMDARI INTERNSHIP/Veltrix Segmentation/veltrix-segmentation_A/data/clean/property_views.parquet

Wrote 11 small tables + 2 large tables to c:\Users\pc\Documents\AMDARI INTERNSHIP\Veltrix Segmentation\veltrix-segmentation_A\data\clean


## 6. Validate
Every line should pass. If one does not, fix the step above, not the check.

In [7]:
c = pd.read_parquet(CLEAN / "customers.parquet")
l = pd.read_parquet(CLEAN / "leads.parquet")

results = {
    "raw customers - duplicates dropped = clean customers": len(raw["customers"]) - len(remap) == len(c),
    "customer_id is unique in clean customers": c["customer_id"].is_unique,
    "every lead's customer_id exists in clean customers": l["customer_id"].isin(c["customer_id"]).all(),
    "leads row count unchanged": len(l) == len(raw["leads"]),
    "14 clean lead sources": l["lead_source"].nunique() == 14,
    "no unmapped country labels": c["country_iso"].notna().all(),
    "no duplicate id remains in any re-pointed table": not any(
        pd.read_parquet(CLEAN / f"{t}.parquet", columns=["customer_id"])["customer_id"].isin(remap).any()
        for t in ["leads", "transactions", "campaign_responses"] if "customer_id" in clean[t].columns),
}
for k, v in results.items():
    print(("PASS  " if v else "FAIL  ") + k)

web_ok = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{(CLEAN / 'web_sessions.parquet').as_posix()}') x JOIN dupmap m ON x.customer_id = m.duplicate_id").df().iloc[0, 0]
print(("PASS  " if web_ok == 0 else "FAIL  ") + "no duplicate id remains in web_sessions")
print("\nrows: raw -> clean")
display(pd.DataFrame({"raw": {t: len(raw[t]) for t in SMALL}, "clean": {t: len(clean[t]) for t in SMALL}}))

PASS  raw customers - duplicates dropped = clean customers
PASS  customer_id is unique in clean customers
PASS  every lead's customer_id exists in clean customers
PASS  leads row count unchanged
PASS  14 clean lead sources
PASS  no unmapped country labels
PASS  no duplicate id remains in any re-pointed table
PASS  no duplicate id remains in web_sessions

rows: raw -> clean


,raw,clean
agents,300,300
campaigns,270,270
campaign_responses,808953,808953
customers,102000,100573
interactions,532789,532789
leads,95804,95804
locations,36,36
payments,106199,106199
properties,16667,16667
site_visits,233593,233593


## Hand-off notes
- `data/clean/*.parquet`: cleaned tables. Notebooks 04+ (features, clustering) should load from here, not from `data/raw/`.
- `outputs/cleaning_log.csv`: what changed and how many rows. Share it for the DA data quality log.
- Add `data/clean/` and `notebooks/outputs/` to `.gitignore` before you commit.

In [8]:
s = cust["stated_budget_gbp"].astype("float64").dropna()
print(pd.cut(s, [0, 1_000, 10_000, 50_000, 100_000, 1_000_000, 1_500_000, 2_440_000, 3_000_000, np.inf], right=False).value_counts().sort_index())

stated_budget_gbp
[0.0, 1000.0)                63
[1000.0, 10000.0)             2
[10000.0, 50000.0)            0
[50000.0, 100000.0)         306
[100000.0, 1000000.0)     48187
[1000000.0, 1500000.0)      819
[1500000.0, 2440000.0)      118
[2440000.0, 3000000.0)       12
[3000000.0, inf)             43
Name: count, dtype: int64
